# AI Risk Control Pipeline — Real GPT-2 Audit (Colab)

This notebook runs the **same pipeline** as the repo, but with the **real pretrained `gpt2`**
backend (`--model gpt2`) instead of the offline TinyGPT toy, and compares the two.

It tests the central hypothesis from the README: *does a pretrained model's representation
generalize to unseen phrasing better than a from-scratch toy?* — i.e. higher leave-one-pair-out
probe accuracy.

> **Not executed in the repo build.** GPT-2 weights download from HuggingFace, which the build
> environment can't reach. Run this top-to-bottom in Colab (CPU is fine; gpt2 is ~0.5 GB).
> The first time, eyeball the sanity-check cell — if shapes/hook look right, the rest follows.


## 1. Setup
Clone your repo (edit the URL) or upload the folder, then install deps.

In [ ]:
# Option A: clone from GitHub (edit to your repo URL)
!git clone https://github.com/YOUR_USERNAME/ai_risk_control_pipeline.git
%cd ai_risk_control_pipeline

# Option B: if you uploaded the folder to Colab instead, just %cd into it and skip the clone.

!pip -q install torch transformers matplotlib numpy

## 2. Sanity-check the GPT-2 adapter
Build gpt2 once, confirm the probe separates a register, and confirm the steering hook actually
moves the representation. (`GPT2Audit` is untested offline, so verify here before the full run.)

In [ ]:
import torch
import model as M, probes as P, weight_attack as WA, activation_steering as AS, behavior_eval as BE
from run_experiment import run_audit

gpt2, axes = M.build_model("gpt2", "risk_dimensions.json", seed=0)
print("layers:", gpt2.n_layers, "| hidden:", gpt2.hidden_dim)   # gpt2: 12 layers, 768 dim

ax = "deception"
probe = P.build_probe(gpt2, axes[ax]["risky"], axes[ax]["safe"])
r = float((gpt2.represent(axes[ax]["risky"]) @ probe).mean())
s = float((gpt2.represent(axes[ax]["safe"])  @ probe).mean())
print(f"[probe] risky={r:+.3f}  safe={s:+.3f}  separation={r-s:+.3f}  (want clearly > 0)")

# does the steering hook change the representation?
vec = AS.safe_direction(gpt2, axes, ax, layer=4)
base = gpt2.represent(axes[ax]["safe"])
stee = gpt2.represent(axes[ax]["safe"], steer=(4, vec * 8.0))
print(f"[hook] mean delta from steering = {(stee-base).norm(dim=1).mean():.3f}  (want > 0)")

## 3. Tune the operating point for GPT-2
GPT-2's residual norms differ from TinyGPT, so the injection `beta` and `steer_scale` need
re-tuning. We grid-search on one axis for a clean operating point: clear drift, ARR in ~0.5–0.9,
no overshoot (steered stays below baseline).

In [ ]:
INJ, STEER = 6, 4   # middle-ish layers for a 12-layer model
safe_pop = [x for a in axes for x in axes[a]["safe"]]
align = lambda mdl, pr, steer=None: -float((mdl.represent(safe_pop, steer=steer) @ pr).mean())

ax = "deception"; pr = P.build_probe(gpt2, axes[ax]["risky"], axes[ax]["safe"])
b = align(gpt2, pr)
print(f"baseline alignment = {b:+.3f}\n")
print(f"{'beta':>5}{'scale':>7}{'inj':>8}{'steer':>8}{'ARR':>8}")
for beta in [4, 8, 16, 32]:
    pm = WA.inject(gpt2, axes, ax, INJ, beta); p = align(pm, pr)
    svec = AS.safe_direction(gpt2, axes, ax, STEER)
    for scale in [4, 8, 16]:
        st = align(pm, pr, steer=(STEER, svec * scale))
        arr = AS.alignment_recovery_rate(b, p, st)
        print(f"{beta:>5}{scale:>7}{p:>8.2f}{st:>8.2f}{100*arr:>7.0f}%")
# pick BETA/SCALE from the table above (clear drift, ARR ~0.5-0.9, steered < baseline)
BETA, SCALE = 16, 8

## 4. Full multi-dimensional audit on GPT-2
Writes artifacts to `results_gpt2/` so the committed TinyGPT `results/` are preserved for comparison.

In [ ]:
cfg = dict(model="gpt2", seed=0, beta=BETA, steer_scale=SCALE, inj_layer=INJ, steer_layer=STEER)
gpt2_out = run_audit(gpt2, axes, cfg, outdir="results_gpt2",
                     kl_betas=(4, 8, 16, 32, 64))

## 5. Compare TinyGPT (toy) vs GPT-2 (real)
The headline comparison is **held-out generalization**: pretrained representations should classify
unseen phrasings better than a from-scratch toy.

In [ ]:
import json, csv
import matplotlib.pyplot as plt
import numpy as np

def load(d):
    g = json.load(open(f"{d}/probe_generalization.json"))["leave_one_pair_out_accuracy_pct"]
    s = json.load(open(f"{d}/steering_results.json"))["axes"]
    b = json.load(open(f"{d}/behavior_change.json"))["behavior"]
    return g, s, b

g_tiny, s_tiny, b_tiny = load("results")        # committed TinyGPT
g_gpt2, s_gpt2, b_gpt2 = load("results_gpt2")   # this run
names = list(g_tiny)

print(f"{'axis':14s}{'gen TinyGPT':>14s}{'gen GPT-2':>12s}{'ARR Tiny':>10s}{'ARR GPT-2':>11s}")
for ax in names:
    print(f"{ax:14s}{g_tiny[ax]['mean']:>12.1f}% {g_gpt2[ax]['mean']:>10.1f}% "
          f"{s_tiny[ax]['ARR_pct']:>8.0f}% {s_gpt2[ax]['ARR_pct']:>9.0f}%")

fig, ax_ = plt.subplots(1, 2, figsize=(12, 4.2))
x = np.arange(len(names)); w = 0.38
ax_[0].bar(x - w/2, [g_tiny[a]["mean"] for a in names], w, label="TinyGPT (toy)", color="#bbb")
ax_[0].bar(x + w/2, [g_gpt2[a]["mean"] for a in names], w, label="GPT-2 (real)", color="#457b9d")
ax_[0].axhline(50, ls="--", color="k", lw=.6, label="chance")
ax_[0].set_xticks(x); ax_[0].set_xticklabels([a[:5] for a in names], rotation=30)
ax_[0].set_ylabel("held-out accuracy %"); ax_[0].set_title("Probe generalization: toy vs real")
ax_[0].legend(fontsize=8)

for tag, beh, c in [("TinyGPT", b_tiny, "#bbb"), ("GPT-2", b_gpt2, "#457b9d")]:
    cu = beh[names[0]]["output_kl_vs_injection_strength"]
    xs = sorted(int(k) for k in cu)
    ax_[1].plot(xs, [cu[str(k)] for k in xs], marker="o", label=tag, color=c)
ax_[1].set_xlabel("injection strength (beta)"); ax_[1].set_ylabel("output KL")
ax_[1].set_title(f"Behaviour change ({names[0]})"); ax_[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig("visualizations/compare_tiny_vs_gpt2.png", dpi=130, bbox_inches="tight")
plt.show()

## 6. Interpretation

- **Generalization:** if GPT-2's held-out accuracy clears TinyGPT's, that is the evidence that the
  mechanism rides on *real* learned representations, not a memorized toy — the answer to *"does
  this generalize to unseen concepts?"* and *"does it scale?"*.
- **Behaviour:** the output-KL curve confirms the injection moves a real model's output
  distribution, monotonically with strength — *"did behaviour actually change?"* → yes, on a real
  model.
- **Honest caveats:** `beta`/`scale`/layers are re-tuned here for GPT-2 and are disclosed in
  `results_gpt2/steering_results.json`. Numbers depend on them. Commit `results_gpt2/` and the
  comparison figure alongside the TinyGPT results so the repo shows the toy→real progression.
